# Fine-tune Audio Classification with Wav2Vec2

This notebook demonstrates how to fine-tune the facebook/wav2vec2-base model for audio classification tasks. Wav2Vec2 is a self-supervised pre-trained model that learns powerful audio representations, making it highly effective for downstream tasks like keyword spotting, speech command recognition, and audio classification.

We'll use the Hugging Face Transformers library to:
- Load a pre-trained Wav2Vec2 model
- Fine-tune it on an audio classification dataset
- Evaluate the model's performance
- Run inference on new audio samples

In [ ]:
!pip install transformers datasets accelerate evaluate scikit-learn

## Load Dataset

In [ ]:
from datasets import load_dataset, Audio
import numpy as np

# Load the SUPERB keyword spotting dataset (speech_commands subset)
dataset = load_dataset("superb", "ks", trust_remote_code=True)

# Use smaller subsets for demo (increase for real training)
dataset["train"] = dataset["train"].select(range(1000))
dataset["validation"] = dataset["validation"].select(range(200))
dataset["test"] = dataset["test"].select(range(200))

# Inspect the dataset structure
print("Dataset structure:")
print(dataset)

# Get unique labels
labels = dataset["train"].features["label"].names
print(f"\nNumber of labels: {len(labels)}")
print(f"Labels: {labels}")

# Create label mappings
label2id = {label: i for i, label in enumerate(labels)}
id2label = {i: label for i, label in enumerate(labels)}

## Load Feature Extractor and Model

In [ ]:
from transformers import AutoFeatureExtractor, AutoModelForAudioClassification

# Model checkpoint
model_checkpoint = "facebook/wav2vec2-base"

# Load feature extractor
feature_extractor = AutoFeatureExtractor.from_pretrained(model_checkpoint)
print(f"Feature extractor sampling rate: {feature_extractor.sampling_rate} Hz")

# Load model for audio classification
num_labels = len(labels)
model = AutoModelForAudioClassification.from_pretrained(
    model_checkpoint,
    num_labels=num_labels,
    label2id=label2id,
    id2label=id2label,
    ignore_mismatched_sizes=True
)

print(f"\nModel loaded with {num_labels} labels")
print(f"Model config: {model.config}")

## Preprocess Data

In [ ]:
# Resample audio to match feature extractor's expected sampling rate
dataset = dataset.cast_column("audio", Audio(sampling_rate=feature_extractor.sampling_rate))

# Preprocessing function
def preprocess_function(examples):
    """Extract audio features from raw audio arrays."""
    audio_arrays = [x["array"] for x in examples["audio"]]
    inputs = feature_extractor(
        audio_arrays,
        sampling_rate=feature_extractor.sampling_rate,
        max_length=16000,  # 1 second at 16kHz
        truncation=True,
        padding=True,
    )
    return inputs

# Apply preprocessing to all splits
print("Preprocessing dataset...")
encoded_dataset = dataset.map(
    preprocess_function,
    remove_columns=["audio", "file"],
    batched=True,
    batch_size=100,
    num_proc=1,
)

print("\nPreprocessed dataset:")
print(encoded_dataset)
print(f"\nSample features shape: {np.array(encoded_dataset['train'][0]['input_values']).shape}")

## Training

In [ ]:
from transformers import TrainingArguments, Trainer
import evaluate

# Load accuracy metric
accuracy_metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    """Compute accuracy metrics for evaluation."""
    predictions = np.argmax(eval_pred.predictions, axis=1)
    references = eval_pred.label_ids
    accuracy = accuracy_metric.compute(predictions=predictions, references=references)
    return accuracy

# Training arguments
training_args = TrainingArguments(
    output_dir="./wav2vec2-audio-classification",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=3e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=2,  # Small number for demo; increase for real training
    warmup_ratio=0.1,
    logging_steps=10,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    push_to_hub=False,
    report_to="none",
)

# Initialize Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=encoded_dataset["train"],
    eval_dataset=encoded_dataset["validation"],
    processing_class=feature_extractor,
    compute_metrics=compute_metrics,
)

# Start training
print("Starting training...")
train_result = trainer.train()

# Print training metrics
print("\nTraining completed!")
print(f"Training metrics: {train_result.metrics}")

## Evaluate

In [ ]:
from sklearn.metrics import classification_report

# Evaluate on test set
print("Evaluating on test set...")
eval_results = trainer.evaluate(encoded_dataset["test"])
print(f"\nTest set metrics: {eval_results}")

# Get predictions for classification report
predictions = trainer.predict(encoded_dataset["test"])
predicted_labels = np.argmax(predictions.predictions, axis=1)
true_labels = predictions.label_ids

# Print detailed classification report
print("\nClassification Report:")
print("=" * 80)
report = classification_report(
    true_labels,
    predicted_labels,
    target_names=labels,
    digits=4
)
print(report)

## Inference

In [ ]:
from transformers import pipeline

# Create audio classification pipeline
classifier = pipeline(
    "audio-classification",
    model=model,
    feature_extractor=feature_extractor
)

# Get a sample from the test set
sample_idx = 0
sample = dataset["test"][sample_idx]
audio_array = np.array(sample["audio"]["array"])
print(f"Sample audio info:")
print(f"  - Sampling rate: {sample['audio']['sampling_rate']} Hz")
print(f"  - Array shape: {audio_array.shape}")
print(f"  - True label: {labels[sample['label']]}")

# Run inference
print("\nRunning inference...")
predictions = classifier(audio_array)

# Display top predictions
print("\nTop 5 predictions:")
print("=" * 50)
for i, pred in enumerate(predictions[:5], 1):
    print(f"{i}. {pred['label']:20s} - Score: {pred['score']:.4f}")

# Check if prediction is correct
predicted_label = predictions[0]["label"]
true_label = labels[sample["label"]]
is_correct = predicted_label == true_label
print(f"\nPrediction {'CORRECT' if is_correct else 'INCORRECT'}!")
print(f"Predicted: {predicted_label}, True: {true_label}")